In [ ]:
!pip install --quiet -U langchain-community langchain-experimental langchain_google_genai

In [ ]:
!pip --quiet install -U langchain-core langgraph langgraph-prebuilt

In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from typing import Annotated
from langgraph.graph.message import add_messages

# Multi-Agent Systems with LangChain & LangGraph (2-Hour Teaching Guide)

Welcome to this comprehensive guide on building **Multi-Agent Systems** using **LangChain**, **LangGraph**, and Google's **Gemini LLM**.

### Learning Objectives
By the end of this session, students will understand:
1. **The Multi-Agent Paradigm**: Why and when to divide tasks among specialized agents rather than relying on a single monolithic prompt.
2. **State Management**: How LangGraph manages shared, mutable state across multiple agents.
3. **Router Patterns**: How to design conditions for passing execution control between agents.
4. **Practical Implementation**: Building, visualizing, and running an end-to-end multi-agent workflow.

## Step 1: Install Dependencies & Setup Environment

First, we install the necessary packages.
- `langgraph` is used for stateful, multi-actor applications.
- `langchain-google-genai` provides the integration for Google's Gemini LLMs.
- `langchain-core` provides standard abstractions like messages and chat models.

### API Key Configuration
Before proceeding, ensure you have a `GEMINI_API_KEY` configured. We will retrieve it securely from Google Colab's user secrets.

In [ ]:
import os
from google.colab import userdata

# Retrieve Gemini API Key from Colab Secrets
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("gemini_key")
    print("GEMINI_API_KEY successfully loaded from Colab Secrets!")
except Exception as e:
    print("Warning: GEMINI_API_KEY not found in secrets. Please add it to your Colab Secrets (🔑 icon) with the exact name 'GEMINI_API_KEY'.")

GEMINI_API_KEY successfully loaded from Colab Secrets!


# ## Step 2: Architecture & Concept Overview

In our upgraded 4-agent system, we use an **Orchestrator-Worker** design pattern. The Orchestrator acts as the central router deciding what steps to take next based on the workflow state.

```
                    +--------------------+
                    |       START        |
                    +---------+----------+
                              |
                              v
                  +-----------+-----------+
                  |  Orchestrator Router  |<-------------------+
                  +-----+-----+-----+-----+	                   |
                        |     |     |                          |
           +------------+     |     +------------+             |
           |                  |                  |             |
           v                  v                  v             |
    +--------------+   +--------------+   +--------------+     |
    |  Researcher  |   |    Writer    |   |    Editor    |     |
    +------+-------+   +------+-------+   +------+-------+     |
           |                  |                  |             |
           +------------------+------------------+-------------+
                              |
                              v
                           +--+---+
                           | END  |
                           +------+
```

- **Orchestrator Agent**: Analyzes the dialog state and delegates control dynamically.
- **Specialized Worker Nodes**: Execute specific sub-tasks (Research, Writing, Editing) and report back.

## Step 3: Define Agent Helpers and the State

Let's define our state type. In LangGraph, the state is passed around to every node. We will use a TypedDict containing a list of messages. We will use `Annotated` with `add_messages` to ensure new messages from agents are appended to the existing conversation list instead of overwriting it.

In [ ]:
from typing import Sequence
#from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
#from langgraph.graph.message import add_messages

# Define the state of our agent network
class AgentState(TypedDict):
    # add_messages ensures that any returned message list gets appended to the state
    messages: Annotated[Sequence[BaseMessage], add_messages]

# ## Step 4: Define the Specialized Agents (Nodes)

We will define our four agents:
1. **Orchestrator**: Inspects the workflow, determines which specialist to invoke next, or chooses to stop and present the final draft.
2. **Researcher**: Compiles raw facts on the requested topic.
3. **Writer**: Structures facts into an engaging, technical draft.
4. **Editor**: Reviews, formats, and polishes the draft into a professional standard.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# Initialize the Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.7)

# Helper to ensure strict human-AI message alternation for Gemini API
def get_clean_history(messages):
    cleaned = []
    for msg in messages:
        # Skip pure system routing signals like "research", "write", "edit", "finish" or signature blocks
        if isinstance(msg, AIMessage):
            content_str = str(msg.content).strip().lower()
            if content_str in ["research", "write", "edit", "finish"]:
                continue
            if isinstance(msg.content, list) and len(msg.content) > 0:
                # Check if it is a list block containing only the routing keyword
                first_val = msg.content[0]
                if isinstance(first_val, dict) and first_val.get("text", "").strip().lower() in ["research", "write", "edit", "finish"]:
                    continue
        cleaned.append(msg)

    # If the last message is an AI message, Gemini API throws an error because it ends with a model turn.
    # We append a brief virtual HumanMessage to keep the turns alternating.
    if cleaned and isinstance(cleaned[-1], AIMessage):
        cleaned.append(HumanMessage(content="Review the work done above and proceed with the next step according to the plan."))

    return cleaned

# 1. Orchestrator Node (Appends route instructions for conditional routing)
def orchestrator_node(state: AgentState):
    print("--- Executing Orchestrator Agent ---")
    messages = state["messages"]

    system_instruction = SystemMessage(content=(
        "You are the system coordinator. Your task is to analyze the conversation and choose the next action.\n"
        "Available next steps are strictly one of: 'research', 'write', 'edit', or 'finish'.\n\n"
        "Rules:\n"
        "1. If the topic has not been researched yet, choose 'research'.\n"
        "2. If research is complete but no draft exists, choose 'write'.\n"
        "3. If a draft exists but has not been edited, choose 'edit'.\n"
        "4. If the editor has polished the post, choose 'finish'.\n\n"
        "Respond with ONLY the lowercase word of the action: 'research', 'write', 'edit', or 'finish'."
    ))

    # The orchestrator looks at the clean worker outputs
    response = llm.invoke([system_instruction] + get_clean_history(messages))
    return {"messages": [response]}

# 2. Researcher Node
def researcher_node(state: AgentState):
    print("--- Executing Researcher Agent ---")
    messages = state["messages"]

    system_instruction = SystemMessage(content=(
        "You are an expert tech researcher. Gather and list technical facts, historical context, "
        "and key details on the requested topic. Do not write a blog post; write pure, bulleted research notes."
    ))

    # Filter the input to avoid sending trailing Orchestrator AIMessages to the model
    response = llm.invoke([system_instruction] + get_clean_history(messages))
    # We add a clear prefix indicating these are research notes
    raw_response = AIMessage(content=f"[RESEARCH NOTES]\n{response.content}")
    return {"messages": [raw_response]}

# 3. Writer Node
def writer_node(state: AgentState):
    print("--- Executing Writer Agent ---")
    messages = state["messages"]

    system_instruction = SystemMessage(content=(
        "You are a technical writer. Convert the provided [RESEARCH NOTES] into an elegant first-draft article. "
        "Keep it well structured with clear markdown headings, introductions, and explanations."
    ))

    # Filter the input to avoid sending trailing Orchestrator AIMessages to the model
    response = llm.invoke([system_instruction] + get_clean_history(messages))
    raw_response = AIMessage(content=f"[DRAFT]\n{response.content}")
    return {"messages": [raw_response]}

# 4. Editor Node
def editor_node(state: AgentState):
    print("--- Executing Editor Agent ---")
    messages = state["messages"]

    system_instruction = SystemMessage(content=(
        "You are a senior tech editor. Take the [DRAFT] and review it for clarity, conciseness, tone, and grammar. "
        "Generate the final markdown output, ensuring it has flawless readability, formatting, and no metadata tags."
    ))

    # Filter the input to avoid sending trailing Orchestrator AIMessages to the model
    response = llm.invoke([system_instruction] + get_clean_history(messages))
    raw_response = AIMessage(content=f"[FINAL PUBLICATION]\n{response.content}")
    return {"messages": [raw_response]}

# ## Step 5: Construct and Compile the LangGraph WorkFlow

Now, we map out the paths. The Orchestrator will be the entry node, and after each specialist performs their task, the control returns back to the Orchestrator to decide the next action dynamically.

In [ ]:
from langgraph.graph import StateGraph, START, END

# Define the routing logic based on Orchestrator's last message output
def route_next_action(state: AgentState):
    last_message_content = state["messages"][-1].content

    # If the LLM returned structured content blocks (list of dicts)
    if isinstance(last_message_content, list):
        # Extract the text content from the blocks
        text_parts = []
        for block in last_message_content:
            if isinstance(block, dict) and "text" in block:
                text_parts.append(block["text"])
            elif isinstance(block, str):
                text_parts.append(block)
        last_message = "".join(text_parts).strip().lower()
    else:
        last_message = str(last_message_content).strip().lower()

    if "research" in last_message:
        return "researcher"
    elif "write" in last_message:
        return "writer"
    elif "edit" in last_message:
        return "editor"
    else:
        return END

# Initialize building process
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("orchestrator", orchestrator_node)
builder.add_node("researcher", researcher_node)
builder.add_node("writer", writer_node)
builder.add_node("editor", editor_node)

# Start state flows immediately to the Orchestrator
builder.add_edge(START, "orchestrator")

# The Orchestrator makes dynamic conditional transitions
builder.add_conditional_edges(
    "orchestrator",
    route_next_action,
    {
        "researcher": "researcher",
        "writer": "writer",
        "editor": "editor",
        END: END
    }
)

# Workers return control back to Orchestrator to decide next phase
builder.add_edge("researcher", "orchestrator")
builder.add_edge("writer", "orchestrator")
builder.add_edge("editor", "orchestrator")

# Compile the multi-agent graph
app = builder.compile()

## Step 6: Execute the Multi-Agent System

Let's run our compiled graph with an input query.

In [ ]:
# Prepare the initial query
initial_input = {
    "messages": [HumanMessage(content="Explain the concept of quantum computing and how it differs from classical computing.")]
}

# Stream and watch the step-by-step orchestrator coordination
for output in app.stream(initial_input):
    for key, value in output.items():
        print(f"Completed Node: {key}")
        print("-" * 50)
        print(value["messages"][-1].content)
        print("\n")

--- Executing Orchestrator Agent ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Completed Node: orchestrator
--------------------------------------------------
[{'type': 'text', 'text': 'research', 'extras': {'signature': 'EmAKXgFpFH0TWwJTw+f2ZmS22B+uAhbWJUa/CJB/pLWw+S1DddiAuvDI5IbXF/PCi4cpk2rNpSAdIhv59nBYX5Ib2PoBJGYR+e0D1BXNNbZ7Upf9yQ9PotM6k4vunEhXyw4='}}]


--- Executing Researcher Agent ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Completed Node: researcher
--------------------------------------------------
[RESEARCH NOTES]
[{'type': 'text', 'text': '* **Fundamental Unit of Information:**\n  * Classical Computing: Uses **bits**, which represent binary states of strictly `0` or `1`.\n  * Quantum Computing: Uses **qubits** (quantum bits), which can represent `0`, `1`, or any quantum superposition of both states simultaneously.\n\n* **Core Quantum Mechanics Principles:**\n  * **Superposition:** A qubit exists in a linear combination of states until it is measured. Mathematically, $|\\psi\\rangle = \\alpha|0\\rangle + \\beta|1\\rangle$, where $\\alpha$ and $\\beta$ are probability amplitudes.\n  * **Entanglement:** A phenomenon where quantum particles become inextricably linked such that the state of one instantly dictates the state of another, regardless of distance. This allows exponential scaling of processing power.\n  * **Interference:** Constructive or destructive interference is used to amplify correct calcul

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


... (5 outputs left out)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Completed Node: editor
--------------------------------------------------
[FINAL PUBLICATION]
[{'type': 'text', 'text': "# Demystifying Quantum Computing: A Paradigm Shift Beyond Classical Systems\n\nFor decades, classical computers have been the bedrock of technological advancement. From the smartphones in our pockets to massive cloud supercomputers, they process information using the same fundamental principles established in the mid-20th century. However, as we encounter increasingly complex problems in cryptography, molecular modeling, and optimization, classical systems are beginning to hit a computational wall.\n\nEnter **quantum computing**—a revolutionary paradigm that harnesses the laws of quantum mechanics to process information in ways previously thought impossible. \n\nTo understand how quantum computers are poised to transform technology, we must first examine how they fundamentally differ from their classical counterparts.\n\n---\n\n## 1. The Fundamental Unit of Informati

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Completed Node: orchestrator
--------------------------------------------------
[{'type': 'text', 'text': 'finish', 'extras': {'signature': 'EmAKXgFpFH0TZh0qkAFy1U1u4tqTjHvHPYPsFIF/4A9a/Ui6owzyo9YcddgWkO6Uat/VqDZuzlHvpWpZXM841aio03Mt1LRES1Q34CWhRO5sEdLX7La3jvZ/0V7oiI8emQ0='}}]


